# Proyecto: Análisis de Logs de Servidor Web

### Paso 1: Configuración inicial y carga de datos

In [32]:
from pyspark.sql import SparkSession
import os

# TODO: Crear SparkSession con configuración
spark = SparkSession.builder \
    .appName("AnalisisStocks") \
    .config("spark.executor.memory", "512m") \
    .getOrCreate()

# TODO: Obtener SparkContext
sc = spark.sparkContext

# TODO: Construir ruta absoluta
path_relativo = "./DATOS/stocks.txt"
path_absoluto = os.path.abspath(path_relativo)
path_final = "file://" + path_absoluto

# TODO: Cargar archivo
rdd = sc.textFile(path_final)

# TODO: Verificar carga
rdd.take(5)

['id,name,category,quantity,price',
 '1,iPhone,Electronics,10,899.99',
 '2,Macbook,Electronics,5,1299.99',
 '3,iPad,Electronics,15,499.99',
 '4,Samsung TV,Electronics,8,799.99']

In [33]:
## Extra: eliminación de encabezado:
header = rdd.first()
print(f"Encabezado: {header}")

rdd_ne = rdd.filter(lambda linea: linea != header)

rdd_ne.take(5)

Encabezado: id,name,category,quantity,price


['1,iPhone,Electronics,10,899.99',
 '2,Macbook,Electronics,5,1299.99',
 '3,iPad,Electronics,15,499.99',
 '4,Samsung TV,Electronics,8,799.99',
 '5,LG TV,Electronics,10,699.99']

### Paso 2: Limpieza y Parseo de Datos

__Advertencia__: El dataframe obtenido con los comandos concedidos difiere de aquel con base al cual se creó el instructivo de práctica, por lo que se tomarán variables (columnas) diferentes que se ajusten en la medida de lo posible a las indicaciones y operaciones deseadas.

In [34]:
def parsear_linea(linea):
    try: 
        partes = linea.split(",")
        nombre = partes[1].strip()
        categoría = partes[2]
        precio = float(partes[-1])
        cantidad = int(partes[-2])
        return (nombre, categoría, cantidad, precio)
    except: 
        return None

rdd = rdd_ne.map(parsear_linea).filter(lambda x: x is not None)
print(f"Líneas válidas: {rdd.count()}")

Líneas válidas: 30


In [35]:
rdd.first()

('iPhone', 'Electronics', 10, 899.99)

### Paso 3: Análisis por símbolo - Estadísticas básicas

(realizado con par `categoría-precio`)

In [36]:
# TODO: Crear pares (símbolo, precio)
par_rdd = rdd.map(lambda x: (x[1], x[3]))

# TODO: Calcular máximo por símbolo
maximos = par_rdd.reduceByKey(lambda a, b: a if a > b else b)

# TODO: Calcular mínimo por símbolo
mínimos = par_rdd.reduceByKey(lambda a, b: a if a < b else b)

# TODO: Calcular promedio (suma, contador)
# aggregateByKey(valor_inicial, seqFunc, combFunc)
precio_promedio = par_rdd.aggregateByKey(
    (0.0, 0),  # (suma, contador)
    lambda acc, valor: (acc[0] + valor, acc[1] + 1),  # Combinar valor con acumulador
    lambda acc1, acc2: (acc1[0] + acc2[0], acc1[1] + acc2[1])   # Combinar dos acumuladores
).mapValues(lambda x: x[0] / x[1])

print("La lista de precios máximos por categoría")
for cat, pre in sorted(maximos.collect()):
    print(f"{cat}: ${pre:.2f}")

print()

print("La lista de precios mínimos por categoría")
for cat, pre in sorted(mínimos.collect()):
    print(f"{cat}: ${pre:.2f}")

print()

print("La lista de precios promedios por categoría")
for cat, pre in sorted(precio_promedio.collect()):
    print(f"{cat}: ${pre:.2f}")

La lista de precios máximos por categoría
Accessories: $29.99
Clothing: $199.99
Electronics: $1299.99
Food: $5.99
Furniture: $249.99
Sports: $49.99

La lista de precios mínimos por categoría
Accessories: $24.99
Clothing: $14.99
Electronics: $129.99
Food: $0.25
Furniture: $79.99
Sports: $19.99

La lista de precios promedios por categoría
Accessories: $27.49
Clothing: $99.28
Electronics: $586.66
Food: $2.30
Furniture: $141.99
Sports: $34.99


### Paso 4: ANálisis de volumen - Top días de mayor actividad

__Advertencia__: Dado que el dataframe actual no tiene una columna con datos de tiempo en general, se tomará la columna `cantidad` para realizar las operaciones.

In [37]:
# TODO: Crear pares (Categoría, cantidad)
rdd_cc = rdd.map(lambda x: (x[1], x[2]))

# TODO: Sumar cantidad por categoría
sum_cat = rdd_cc.reduceByKey(lambda a, b: a + b)
print("Suma de productos por categoría")
for cat, can in sum_cat.collect():
    print(f"{cat}, {can} unidades")
print()

# TODO: Ordenar por cantidad descendente
desc_cc = sum_cat.sortBy(lambda x: x[1], ascending=False)
print("Suma de categorías por orden descendente")
for cat, can in desc_cc.collect():
    print(f"{cat}, {can} unidades")
print()

# TODO: Mostrar top 3
print("Top 3 categorías con más unidades vendidas:")
for cat, prec in desc_cc.take(3):
    print(f"{cat}: {prec:,} unidades")

Suma de productos por categoría
Clothing, 200 unidades
Furniture, 41 unidades
Sports, 35 unidades
Accessories, 55 unidades
Electronics, 98 unidades
Food, 450 unidades

Suma de categorías por orden descendente
Food, 450 unidades
Clothing, 200 unidades
Electronics, 98 unidades
Accessories, 55 unidades
Furniture, 41 unidades
Sports, 35 unidades

Top 3 categorías con más unidades vendidas:
Food: 450 unidades
Clothing: 200 unidades
Electronics: 98 unidades


### Paso 5: Análisis avanzado - Volatilidad por símbolo

__Advertencia__: Puesto que el dataframe concedido no tiene las variables esperadas, se hará un análisis con `categoría` y `precio` para realizar las operaciones.

In [38]:
# TODO: Crear pares (Categoría, precio)
rdd_cp = rdd.map(lambda x: (x[1], x[3]))

# TODO: Calcular max y min por símbolo en una sola pasada
# Usa aggregateByKey con valor inicial (precio_max, precio_min)
max_min = rdd_cp.aggregateByKey(
    (float('inf'), float('-inf')),  # (max, min)
    lambda acc, precio: (min(acc[0], precio), (max(acc[1], precio))),  # Actualizar max y min
    lambda acc1, acc2: (min(acc1[0], acc2[0]), (max(acc1[1], acc2[1])))    # Combinar dos acumuladores
)

precio_promedio = rdd_cp.aggregateByKey(
    (0.0, 0),  # (suma, contador)
    lambda acc, valor: (acc[0] + valor, acc[1] + 1),  # Combinar valor con acumulador
    lambda acc1, acc2: (acc1[0] + acc2[0], acc1[1] + acc2[1])   # Combinar dos acumuladores
).mapValues(lambda x: x[0] / x[1])


# TODO: Calcular volatilidad
vol = max_min.mapValues(
    lambda x: ((x[1] - x[0]) / x[0]) * 100
)

# TODO: Ordenar por volatilidad
mas_vol = vol.sortBy(lambda x: x[1], ascending=False).take(5)
mas_est = vol.sortBy(lambda x: x[1], ascending=True).take(5)

# TODO: Mostrar resultados
print("categorías con ventas más volátiles:")
for cat, prec in mas_vol:
    print(f"{cat}: {prec:.2f}%")

print()

print("categorías con ventas más estables:")
for cat, prec in mas_est:
    print(f"{cat}: {prec:.2f}%")

categorías con ventas más volátiles:
Food: 2296.00%
Clothing: 1234.16%
Electronics: 900.07%
Furniture: 212.53%
Sports: 150.08%

categorías con ventas más estables:
Accessories: 20.01%
Sports: 150.08%
Furniture: 212.53%
Electronics: 900.07%
Clothing: 1234.16%


### Paso 6: Guardar resultados y Análisis final

In [39]:
rdd.first()

('iPhone', 'Electronics', 10, 899.99)

In [40]:
# TODO: Guardar resultados
# Para archivos locales, usa file:// en la ruta
output_path_volumen = "file://" + os.path.abspath("./RESULTADOS/cantidad_por_categoría")
sum_cat.saveAsTextFile(output_path_volumen)

output_path_stats = "file://" + os.path.abspath("./RESULTADOS/stats_por_categoría")
max_min.saveAsTextFile(output_path_stats)

# TODO: Calcular métricas para resumen
total_registros = rdd.count()
simbolos_unicos = rdd.map(lambda x: x[1]).distinct().count()
simbolo_mayor_precio = precio_promedio.sortBy(lambda x: x[1], ascending=False).first()
dia_mayor_volumen = sum_cat.sortBy(lambda x: x[1], ascending=False).first()
simbolo_mas_volatil = vol.sortBy(lambda x: x[1], ascending=False).first()

# TODO: Mostrar resumen ejecutivo
print("\n" + "="*60)
print("RESUMEN EJECUTIVO - ANÁLISIS DE VENTAS")
print("="*60)
print(f"\nTotal de registros procesados: {total_registros:,}")
print(f"Número de categorías únicos: {simbolos_unicos}")
print(f"\Categoría con mayor precio promedio: {simbolo_mayor_precio[0]} (${simbolo_mayor_precio[1]:.2f})")
print(f"Categoría con mayores ventas: {dia_mayor_volumen[0]} ({dia_mayor_volumen[1]:,} unidades)")
print(f"categoría más volátil: {simbolo_mas_volatil[0]} ({simbolo_mas_volatil[1]:.2f}%)")

# TODO: Cerrar sesión
spark.stop()
print("\n✅ Análisis completado. Sesión cerrada.")


RESUMEN EJECUTIVO - ANÁLISIS DE VENTAS

Total de registros procesados: 30
Número de categorías únicos: 6
\Categoría con mayor precio promedio: Electronics ($586.66)
Categoría con mayores ventas: Food (450 unidades)
categoría más volátil: Food (2296.00%)

✅ Análisis completado. Sesión cerrada.
